# В этом ноутбуке будем резать на чанки  документ

In [2]:
from dotenv import load_dotenv
import os

from matplotlib.font_manager import json_load

In [3]:
load_dotenv()

OPENAI_KEY = os.getenv("OPENAI_API_KEY")

In [4]:
import json
from pathlib import Path

def split_and_export(input_path, output_jsonl_path, max_chars=3000):
    # Загружаем текст
    text = Path(input_path).read_text(encoding="utf-8")
    paragraphs = text.split("\n\n")

    output_jsonl = open(output_jsonl_path, "w", encoding="utf-8")

    current = ""
    chunk_id = 1

    for para in paragraphs:
        if len(current) + len(para) < max_chars:
            current += para + "\n\n"
        else:
            chunk_data = {
                "id": f"chunk_{chunk_id:04}",
                "text": current.strip()
            }

            #Записываем строкой в book.jsonl
            output_jsonl.write(json.dumps(chunk_data, ensure_ascii=False) + "\n")

            current = para + "\n\n"
            chunk_id += 1

    # Записываем последний остаток
    if current.strip():
        chunk_data = {
            "id": f"chunk_{chunk_id:04}",
            "text": current.strip()
        }
        output_jsonl.write(json.dumps(chunk_data, ensure_ascii=False) + "\n")

    output_jsonl.close()
    print(f"✅ Сохранено {chunk_id} чанков")
    print(f"📘 JSONL файл: {output_jsonl_path}")

In [5]:
split_and_export(
    input_path="output/algebra_6_1.md",
    output_jsonl_path="chunked_jsonl/algebra_6_1_from_md_format.jsonl",
    max_chars=1500
)

✅ Сохранено 261 чанков
📘 JSONL файл: chunked_jsonl/algebra_6_1_from_md_format.jsonl


In [6]:
import json
from pathlib import Path
import re

len_of_chunks = []

def split_by_sections(input_path, output_jsonl_path, max_chars_warning = 5000):
    text = Path(input_path).read_text(encoding="utf-8")

    # Разделяем по \section или \section* \section*
    section_splits = re.split(r'\\*section\*?|\\*section', text)

    output_jsonl = open(output_jsonl_path, "w", encoding="utf-8")

    chunk_id = 1
    for section in section_splits:
        cleaned = section.strip()
        if not cleaned:
            continue

        chunk_data = {
            "id": f"chunk_{chunk_id:03}",
            "text": cleaned
        }
        len_of_chunks.append(len(cleaned))
        if len(cleaned) > max_chars_warning:
            print(f"WARNING: chunk[{chunk_id}] превысил MAX_CHARS_WARNING{max_chars_warning} символов на чанк")
            raise Exception(f"ERROR: chunk[{chunk_id}] превысил MAX_CHARS_WARNING{max_chars_warning} символов на чанк")
        # Записываем строкой в jsonl
        output_jsonl.write(json.dumps(chunk_data, ensure_ascii=False) + "\n")

        chunk_id += 1

    output_jsonl.close()
    print(f"INFO: Всего сохранено {chunk_id - 1} чанков в JSONL-файле {output_jsonl_path}")
    print(f"INFO: Максимальное количество символов на чанк: {max(len_of_chunks)}")

    return max(len_of_chunks)


In [ ]:
split_by_sections(
    input_path="output/algebra_6_1.md",
    output_jsonl_path="chunked_jsonl/algebra_6_by_sections.jsonl",
    max_chars_warning=8000
)
print(sorted(len_of_chunks, reverse=True))

In [14]:
import json
from pathlib import Path
import re

len_of_chunks = []

def split_by_str(input_path, output_jsonl_path, split_by, max_chars_warning = 5000):
    text = Path(input_path).read_text(encoding="utf-8")

    # Разделяем по str
    section_splits = re.split(split_by, text)

    output_jsonl = open(output_jsonl_path, "w", encoding="utf-8")

    chunk_id = 1
    for section in section_splits:
        cleaned = section.strip()
        if not cleaned:
            continue

        chunk_data = {
            "id": f"chunk_{chunk_id:03}",
            "text": cleaned
        }
        len_of_chunks.append(len(cleaned))
        if len(cleaned) > max_chars_warning:
            print(f"WARNING: chunk[{chunk_id}] превысил MAX_CHARS_WARNING{max_chars_warning} символов на чанк")
            raise Exception(f"ERROR: chunk[{chunk_id}] превысил MAX_CHARS_WARNING{max_chars_warning} символов на чанк")
        # Записываем строкой в jsonl
        output_jsonl.write(json.dumps(chunk_data, ensure_ascii=False) + "\n")

        chunk_id += 1

    output_jsonl.close()
    print(f"INFO: Всего сохранено {chunk_id - 1} чанков в JSONL-файле {output_jsonl_path}")
    print(f"INFO: Максимальное количество символов на чанк: {max(len_of_chunks)}")

    return max(len_of_chunks)


In [133]:
split_by_str(
    input_path="output/algebra_6_1.md",
    output_jsonl_path="chunked_jsonl/algebra_6_by_str.jsonl",
    split_by = r"#",
    max_chars_warning=10000
)
print(sorted(len_of_chunks, reverse=True))

INFO: Всего сохранено 266 чанков в JSONL-файле chunked_jsonl/algebra_6_by_str.jsonl
INFO: Максимальное количество символов на чанк: 9896
[9896, 9896, 9896, 9896, 9896, 9896, 7961, 7961, 7961, 7961, 7961, 7075, 7075, 7075, 7075, 7075, 6584, 6584, 6584, 6584, 6584, 5491, 5491, 5491, 5491, 5491, 5465, 5465, 5465, 5465, 5465, 5465, 4795, 4795, 4795, 4795, 4795, 4711, 4711, 4711, 4711, 4711, 3857, 3857, 3857, 3857, 3857, 3848, 3848, 3848, 3848, 3848, 3840, 3840, 3840, 3840, 3840, 3839, 3839, 3839, 3839, 3839, 3761, 3761, 3761, 3761, 3761, 3761, 3761, 3761, 3761, 3761, 3694, 3694, 3694, 3694, 3694, 3637, 3637, 3637, 3637, 3637, 3617, 3617, 3617, 3617, 3617, 3391, 3391, 3391, 3391, 3391, 3297, 3297, 3297, 3297, 3297, 3268, 3268, 3268, 3268, 3268, 3268, 3169, 3169, 3169, 3169, 3169, 3078, 3078, 3078, 3078, 3078, 3030, 3030, 3030, 3030, 3030, 3027, 3027, 3027, 3027, 3027, 2992, 2990, 2990, 2990, 2990, 2989, 2987, 2987, 2987, 2987, 2983, 2983, 2983, 2983, 2983, 2977, 2977, 2977, 2977, 2977, 2932

In [232]:
SYSTEM_PROMPT = """
Ты — умный и точный парсер учебников по математике и редактор LaTeX-формул. Твоя задача — извлекать задачи из фрагментов книг и корректно оформлять математические формулы. Ты не изменяешь структуру текста и не придумываешь ничего от себя.


1. Найди все задачи. Задачей считается которое:
   - начинается с номера (например: 1), 25., 122*) или содержит слово "Задача", "Решите", "Вычислите", "Найдите", и т. п.
   - имеет смысл как самостоятельное задание

2. Для каждой задачи выдай:
   - "id": номер задачи или порядковый номер из текста
   - "original": исходный текст задачи, без решения
   - "solution": пошаговое решение, если есть (если нет — null)
   - "answer": окончательный ответ (если есть)
   - "topic": выбери тему из списка тем
   - "difficulty": A - легкий, B - средний, C - сложный.
   - "tags": 3–9 тегов, описывающих суть задачи (ключевые объекты, методы решения, типы данных)
   - "type": "Практическая" или "Теоретическая"
   - "valid": TRUE если задача корректная; FALSE если задача обрезана, бессмысленна, или не может быть решена
   - "remark": (если valid = FALSE) — кратко объясни причину (3–7 слов)

3. Все формулы в LaTeX должны быть:
   - обёрнуты в `$...$` (inline) или `\\[...\\]` (block)
   - `\\frac`, `\\cdot`, `\\text` и прочее должны быть синтаксически корректны


4. Если решение отсутствует — установи "solution": null. Не пытайся его придумывать.

5. Верни результат в формате JSON-массива.

6. Не добавляй никаких комментариев, пояснений или текста вне JSON!!!.

Темы:
- Делимость натуральных чисел
- Совместное выполнение действий с обыкновенными и десятичными дробями
- Нахождение процентов от данного числа
- Нахождение числа по его процентам
- Отношение двух чисел
- Деление в данном отношении
- Процентное отношение двух чисел
- Пропорция. Основное свойство пропорции
- Прямо пропорциональная зависимость
- Обратно пропорциональная зависимость
- Решение задач на проценты способом пропорции
- Масштаб
- Длина окружности. Площадь круга. Шар. Сфера
- Положительные и отрицательные числа
- Координатная прямая. Противоположные числа
- Целые и рациональные числа
- Модуль числа
- Сравнение рациональных чисел
- Сложные задачи на движение по реке
- Сложение и вычитание рациональных чисел
- Умножение и деление рациональных чисел
- Представление и преобразование периодических дробей
- Алгебраические выражения. Переменные
- Раскрытие скобок. Коэффициент
- Подобные слагаемые
- Тождественные преобразования

Если тема не подходит — выбери ближайшую по смыслу.
"""

len(SYSTEM_PROMPT)

2496

In [233]:
output_jsonl_path = "chunked_jsonl/algebra_6_by_str.jsonl"

data = []
with open(output_jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

idx_chunk = 6
print(data[idx_chunk])
text_sample = data[idx_chunk]["text"]
print(f"Размер: {len(text_sample)} символов")

{'id': 'chunk_007', 'text': 'Делимость натуральных чисел\n\nПо записи числа, не выполняя деления, можно установить, делится или не делится это число на другое. Для этого пользуются признаками делимости.\n\nІ. Признак делимости натуральных чисел на 2 , на 5 , на 10 , на 3 и на 9 . Задача 1.\nЗаменив звездочку соответствующей цифрой, заполните таблицу.\n\n|  | Числа, <br> делящиеся <br> на 2 | Числа, <br> делящиеся <br> на 5 | Числа, <br> делящиеся <br> на 10 | Числа, <br> делящиеся <br> на 3 | Числа, <br> делящиеся <br> на 9 |\n| :---: | :---: | :---: | :---: | :---: | :---: |\n| $49^{*}$ |  |  |  |  |  |\n| $83^{*}$ |  |  |  |  |  |'}
Размер: 601 символов


In [234]:
def clean_gpt_output(raw: str) -> str:
    # убираем обёртку ```json ... ```
    if raw.lstrip().startswith("```"):
        raw = re.sub(r"```json\s*", "", raw, flags=re.I).strip()
        raw = re.sub(r"```$", "", raw).strip()
    return raw

In [235]:
import openai
client = openai.OpenAI(api_key=OPENAI_KEY)

def process_chunk(chunk_text):
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            temperature=0.2,
            top_p=1,
            frequency_penalty=0,
            presence_penalty=0,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Вот фрагмент текста из учебника:\n\n{chunk_text}"}
            ],

        )
        res = response.choices[0].message.content.strip()
        return clean_gpt_output(res)
    except Exception as e:
        print(f"❌ Ошибка при обработке чанка: {e}")
        return None

In [236]:
res_str = process_chunk(text_sample)

In [237]:
print(res_str)

[
  {
    "id": "1",
    "original": "Заменив звездочку соответствующей цифрой, заполните таблицу.\n\n|  | Числа, <br> делящиеся <br> на 2 | Числа, <br> делящиеся <br> на 5 | Числа, <br> делящиеся <br> на 10 | Числа, <br> делящиеся <br> на 3 | Числа, <br> делящиеся <br> на 9 |\n| :---: | :---: | :---: | :---: | :---: | :---: |\n| $49^{*}$ |  |  |  |  |  |\n| $83^{*}$ |  |  |  |  |  |",
    "solution": null,
    "answer": null,
    "topic": "Делимость натуральных чисел",
    "difficulty": "A",
    "tags": ["делимость", "натуральные числа", "признаки делимости", "таблица", "заполнение", "цифры"],
    "type": "Практическая",
    "valid": true,
    "remark": null
  }
]


In [238]:
res_str_clear = clean_gpt_output(res_str)
res_str_fixed = res_str_clear.replace("\\\\", "\\")
print(res_str_fixed)

[
  {
    "id": "1",
    "original": "Заменив звездочку соответствующей цифрой, заполните таблицу.\n\n|  | Числа, <br> делящиеся <br> на 2 | Числа, <br> делящиеся <br> на 5 | Числа, <br> делящиеся <br> на 10 | Числа, <br> делящиеся <br> на 3 | Числа, <br> делящиеся <br> на 9 |\n| :---: | :---: | :---: | :---: | :---: | :---: |\n| $49^{*}$ |  |  |  |  |  |\n| $83^{*}$ |  |  |  |  |  |",
    "solution": null,
    "answer": null,
    "topic": "Делимость натуральных чисел",
    "difficulty": "A",
    "tags": ["делимость", "натуральные числа", "признаки делимости", "таблица", "заполнение", "цифры"],
    "type": "Практическая",
    "valid": true,
    "remark": null
  }
]


Заменив звездочку соответствующей цифрой, заполните таблицу.

|          | Числа, <br> делящиеся <br> на 2 | Числа, <br> делящиеся <br> на 5 | Числа, <br> делящиеся <br> на 10 | Числа, <br> делящиеся <br> на 3 | Числа, <br> делящиеся <br> на 9 |
|:--------:|:-------------------------------:| :---: | :---: | :---: | :---: |
| $49^{*}$ |                                 |  |  |  |  |
| $83^{*}$ |                                 |  |  |  |  |

|### Парсим обратно, проверяем все ли окей

\[\begin{aligned}& \frac{3}{6}=\frac{45}{x} ; & x=\frac{6 \cdot 45}{3} ; \\& x=90(\text { м). }\end{aligned}\]

1) Как называются числа $a$ и $d$ в пропорции $a: b=c: d$ ?

Проверьте, какие из равенств являются пропорциями (устно):\n1) $8: 2=0,4: 1$;\n2) $\frac{1}{4}=\frac{0,2}{0,8}$;\n3) $7: 0,1=21: 0,3$;\n4) $\frac{9}{2}=\frac{2,7}{0,6}$;\n5) $42: 6=1: \frac{1}{7}$;\n6) $\frac{5}{2}=\frac{0,5}{0,02}$ ?

In [239]:
res_json = json.loads(res_str_fixed)

In [240]:
def prepare_json(dirty_json: list[str]):

    dirty_json[0] = dirty_json[0].removeprefix("[")

    if len(dirty_json) > 1:
        for i in range(len(dirty_json)-1):
            dirty_json[i] += "}"

    dirty_json[-1] = dirty_json[-1].removesuffix("]")

    return dirty_json


In [241]:
prepare_json(["[{123}]"])

['{123}']

In [242]:
prepare_json(["[{123", "{213}", "{312}]"])

['{123}', '{213}}', '{312}']

In [243]:
def process_jsonl_chunks(input_jsonl_path, output_jsonl_path, start_chunk=0, end_chunk=11):
    with open(input_jsonl_path, "r", encoding="utf-8") as infile, \
         open(output_jsonl_path, "w", encoding="utf-8") as outfile:

        i_chunk = 0
        for line in infile:

            if i_chunk >= end_chunk:
                break
            data = json.loads(line)
            chunk_id = data["id"]
            chunk_text = data["text"]

            print(f"Обработка {chunk_id}...")
            if i_chunk >= start_chunk:
                tagged_result = process_chunk(chunk_text)
                if "[]" not in tagged_result:
                    print(tagged_result)
                    print(50 * "~")
                    tagged_result = tagged_result.split("},")
                    tagged_result = prepare_json(tagged_result)
                    for elem in tagged_result:
                        outfile.write(json.dumps(elem, ensure_ascii=False) + "\n")
                    print(f"Сохранено {chunk_id}")
                    print(tagged_result)
                    for i in range(len(tagged_result)):
                        print(50 * "+")
                        print(tagged_result[i])
                else:
                    print(f"Пропущен {chunk_id}")

            i_chunk+=1

        print(f"Все загружено!")


In [244]:
process_jsonl_chunks(
    input_jsonl_path="chunked_jsonl/algebra_6_by_str.jsonl",            # где лежат чанки
    output_jsonl_path="parsed_task_dir/algebra_6_parsed_task_from_md_2.jsonl",
    start_chunk=0,
    end_chunk=20# куда сохранить размеченные задачи
)

Обработка chunk_001...
Пропущен chunk_001
Обработка chunk_002...
Пропущен chunk_002
Обработка chunk_003...
Пропущен chunk_003
Обработка chunk_004...
Пропущен chunk_004
Обработка chunk_005...
Пропущен chunk_005
Обработка chunk_006...
Пропущен chunk_006
Обработка chunk_007...
[
  {
    "id": "1",
    "original": "Заменив звездочку соответствующей цифрой, заполните таблицу.\n\n|  | Числа, <br> делящиеся <br> на 2 | Числа, <br> делящиеся <br> на 5 | Числа, <br> делящиеся <br> на 10 | Числа, <br> делящиеся <br> на 3 | Числа, <br> делящиеся <br> на 9 |\n| :---: | :---: | :---: | :---: | :---: | :---: |\n| $49^{*}$ |  |  |  |  |  |\n| $83^{*}$ |  |  |  |  |  |",
    "solution": null,
    "answer": null,
    "topic": "Делимость натуральных чисел",
    "difficulty": "A",
    "tags": ["делимость", "натуральные числа", "признаки делимости", "таблица", "заполнение", "цифры", "числа с пропусками"],
    "type": "Практическая",
    "valid": true,
    "remark": null
  }
]
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Заменив звездочку соответствующей цифрой, заполните таблицу.

|  | Числа, <br> делящиеся <br> на 2 | Числа, <br> делящиеся <br> на 5 | Числа, <br> делящиеся <br> на 10 | Числа, <br> делящиеся <br> на 3 | Числа, <br> делящиеся <br> на 9 |
| :---: | :---: | :---: | :---: | :---: | :---: |
| $49^{*}$ |  |  |  |  |  |
| $83^{*}$ |  |  |  |  |  |


"Заменив звездочку соответствующей цифрой, запишите:

1) число, кратное 9 :
    *67; $\quad 2 * 9 ; \quad 87 * ; \quad 8 * 2 ; \quad 9 * 6 ; \quad 46^{*}$;

2) наименьшее число, кратное 3 :
    $1 * 0 ; \quad 2 * 1 ; \quad 35^{*} ; \quad * 13 ; \quad 4 * 5 ; \quad 83^{*}$.",


"Найдите наибольший общий делитель числителя и знаменателя и сократите дробь:\n1) $\frac{24}{60}$, НОД $(24,60)=\square$;\n2) $\frac{45}{105}$, НОД $(45,105)=\square$;\n3) $\frac{39}{130}$, НОД $(39,130)=\square$;\n4) $\frac{64}{144}$, НОД $(64,144)=\square$.",


"Какие фигуры, изображенные на рисунке 1 , можно нарисовать одним росчерком (не проведя ни одной линии дважды и не отрывая карандаш от тетради), а какие - нельзя? Перечертите фигуры, которые можно обвести одним росчерком, в тетрадь.
a)
![](https://cdn.mathpix.com/cropped/2025_07_18_666568deb2b7d5cb275bg-007.jpg?height=322&width=379&top_left_y=197&top_left_x=226)
б)
![](https://cdn.mathpix.com/cropped/2025_07_18_666568deb2b7d5cb275bg-007.jpg?height=344&width=304&top_left_y=178&top_left_x=770)


Рис. 1
![](https://cdn.mathpix.com/cropped/2025_07_18_666568deb2b7d5cb275bg-007.jpg?height=357&width=191&top_left_y=171&top_left_x=1202)",


"Задача. Выпишите дроби, которые можно представить в виде десятичных дробей. Запишите их в виде десятичных дробей:\n$3 \frac{1}{5}$;\n$\frac{5}{6} ;$\n$\frac{7}{20} ;$\n$4 \frac{2}{1}$",


## FastAPI vs Flask: что выбрать для микросервисов генерации задач

**FastAPI** и **Flask** — популярные Python-фреймворки для создания API. Вот их сравнение применительно к вашему проекту:

### FastAPI — плюсы:
- **Автоматическая документация (Swagger/OpenAPI)**: FastAPI генерирует Swagger UI "из коробки" — удобно тестировать и документировать API.
- **Валидация данных**: встроенная поддержка Pydantic для строгой проверки входных/выходных данных (JSON, формы и т.д.).
- **Асинхронность**: поддержка async/await, что важно для работы с внешними сервисами (S3, MongoDB, OpenAI API) и высокой нагрузки.
- **Скорость**: FastAPI быстрее Flask за счет асинхронности и оптимизаций.
- **Современный синтаксис**: аннотации типов, автогенерация схем, меньше шаблонного кода.

### Flask — плюсы:
- **Простота**: минималистичный, легко стартовать, много обучающих материалов.
- **Гибкость**: можно строить как простые, так и сложные приложения, но многое нужно реализовывать вручную.
- **Большое сообщество**: много расширений, но для асинхронности и валидации нужны сторонние библиотеки.

### Для вашего проекта:
- **MongoDB**: FastAPI отлично работает с асинхронными драйверами (motor), Flask — только синхронно.
- **S3/Minio**: асинхронные клиенты (aiobotocore) проще интегрировать с FastAPI.
- **Swagger**: FastAPI генерирует автоматически, Flask — только через расширения (flasgger, flask-restx).
- **Микросервисы**: FastAPI лучше подходит для современных микросервисов с высокой нагрузкой и большим количеством API-эндпоинтов.

**Вывод:**  
Для микросервисной архитектуры с MongoDB, S3 и автоматической документацией лучше выбрать FastAPI. Flask подойдет для простых сервисов или если нужен только минимальный API без асинхронности.
